# Notebook 04 - Mini Data Warehouse

## Objectif

Ici, on construit un **DWH volontairement petit**, mais crédible.

## Règle de cadrage

On ne part pas de la source.  
On part des KPI.

### KPI à servir

- nombre d'avis par jour
- délai moyen par type de marché
- top acheteurs
- top CPV principal

### Donc il nous faut :

#### Dimensions
- `dim_date`
- `dim_buyer`
- `dim_market_type`
- `dim_cpv`

#### Facts
- `fact_notice_publication`
- `fact_notice_cpv`

## Pourquoi ce modèle est réaliste

Un DWH doit mettre la donnée **au bon endroit**, **avec la bonne granularité** et de façon **diffusable**. 

Ici :

- `fact_notice_publication` a pour grain **1 ligne = 1 avis**
- `fact_notice_cpv` gère la relation multi-valuée **1 avis -> plusieurs CPV**


In [1]:
# Nous importons pandas.
import pandas as pd

# Nous importons la bibliothèque pathlib pour manipuler les chemins de fichiers proprement.
from pathlib import Path

In [2]:
# Nous définissons un chemin de départ qui correspond au dossier courant du notebook.
current_dir = Path.cwd().resolve()

# Si le notebook est exécuté depuis le dossier du projet, ce test sera vrai.
if (current_dir / "data").exists():
    # Dans ce cas, le dossier courant est déjà la racine du projet.
    project_root = current_dir
else:
    # Sinon, on suppose que le notebook est exécuté depuis le dossier notebooks.
    project_root = current_dir.parent

# Nous définissons ici le dossier qui contient les lots bruts.
data_dir = project_root / "data" / "raw_batches"

# Nous définissons ici le dossier de sortie.
output_dir = project_root / "outputs"

# Nous affichons la racine retenue pour éviter toute ambiguïté.
print("project_root =", project_root)

# Nous affichons la liste des fichiers bruts disponibles.
print("raw files =", sorted(p.name for p in data_dir.glob("*.json")))

project_root = D:\Ens-EPSI\boamp_etl_dwh_tiny_explained_project
raw files = ['B20150308_01_initial.json', 'B20150309_01_incremental.json', 'B20150309_01_rerun.json']


## Étape 1 - Charger les tables de transformation

In [3]:
# Nous lisons la base analytique minimale des avis.
tr_notice_kpi_base = pd.read_csv(output_dir / "transformations" / "tr_notice_kpi_base.csv")

# Nous lisons la table des CPV transformés.
tr_notice_cpv = pd.read_csv(output_dir / "transformations" / "tr_notice_cpv.csv")

# Nous convertissons la date de publication en vrai type date/temps.
tr_notice_kpi_base["publication_date"] = pd.to_datetime(tr_notice_kpi_base["publication_date"])

# Nous affichons les tables chargées.
display(tr_notice_kpi_base)
display(tr_notice_cpv)

,idweb,batch_id,publication_date,buyer_name,market_type_code,response_delay_days,has_deadline_flag,url_avis,objet_court,notice_count
0,15-31501,B20150308_01_initial,2015-03-03,Cté Cnes Moyenne Vilaine et Semnon,TRAVAUX,48,True,https://www.boamp.fr/pages/avis/?q=idweb:15-31501,concession d'aménagement pour la réalisation d...,1
1,15-40000,B20150308_01_initial,2015-03-04,Ville de Rennes,TRAVAUX,14,True,https://www.boamp.fr/pages/avis/?q=idweb:15-40000,maintenance de bâtiments scolaires et travaux ...,1
2,15-40001,B20150308_01_initial,2015-03-05,Ville de Nantes,FOURNITURES,20,True,https://www.boamp.fr/pages/avis/?q=idweb:15-40001,fourniture de matériel informatique pour les s...,1
3,15-40002,B20150308_01_initial,2015-03-06,Conseil départemental de Loire-Atlantique,SERVICES,14,True,https://www.boamp.fr/pages/avis/?q=idweb:15-40002,prestations de nettoyage des locaux administra...,1
4,15-40003,B20150309_01_incremental,2015-03-07,Métropole de Lyon,TRAVAUX,23,True,https://www.boamp.fr/pages/avis/?q=idweb:15-40003,travaux d'aménagement de voirie et signalisation,1
5,15-40004,B20150309_01_incremental,2015-03-08,Communauté urbaine de Bordeaux,SERVICES,14,True,https://www.boamp.fr/pages/avis/?q=idweb:15-40004,services d'assistance juridique en urbanisme,1


,idweb,cpv_code,is_primary_flag
0,15-31501,45111291,True
1,15-31501,71000000,False
2,15-31501,70122100,False
3,15-40000,45453100,True
4,15-40000,50700000,False
5,15-40001,30200000,True
6,15-40001,48000000,False
7,15-40002,90910000,True
8,15-40003,45233140,True
9,15-40003,34992200,False


## Étape 2 - Définir deux fonctions de construction de dimensions

Nous utilisons deux fonctions parce qu'elles clarifient la logique :

- une pour la dimension temps,
- une pour les dimensions simples à clé substitut.

Chaque fonction est courte et documentée.

In [4]:
# Nous définissons une fonction pour construire la dimension date.
def build_dim_date(publication_dates: pd.Series) -> pd.DataFrame:
    """
    Construire une petite dimension date à partir des dates de publication.

    Paramètres
    ----------
    publication_dates : pandas.Series
        Série de dates de publication.

    Retour
    ------
    pandas.DataFrame
        Dimension date avec clé technique et attributs calendaires de base.
    """

    # Nous convertissons la série en timestamps pandas.
    dates = pd.to_datetime(publication_dates)

    # Nous supprimons les valeurs manquantes puis nous normalisons au jour.
    dates = dates.dropna().dt.normalize()

    # Nous extrayons les dates uniques et triées.
    unique_dates = pd.Series(sorted(dates.unique()))

    # Nous créons le DataFrame de base.
    dim = pd.DataFrame({"full_date": unique_dates})

    # Nous construisons une clé date au format AAAAMMJJ.
    dim["date_key"] = dim["full_date"].dt.strftime("%Y%m%d").astype(int)

    # Nous ajoutons les attributs calendaires utiles.
    dim["year"] = dim["full_date"].dt.year
    dim["month"] = dim["full_date"].dt.month
    dim["day"] = dim["full_date"].dt.day
    dim["month_label"] = dim["full_date"].dt.strftime("%Y-%m")

    # Nous retournons les colonnes dans un ordre lisible.
    return dim[["date_key", "full_date", "year", "month", "day", "month_label"]]


# Nous définissons une fonction générique de dimension simple.
def build_simple_dimension(values: pd.Series, key_name: str, value_name: str) -> pd.DataFrame:
    """
    Construire une dimension simple avec clé substitut.

    Paramètres
    ----------
    values : pandas.Series
        Série des valeurs métier.
    key_name : str
        Nom de la clé substitut.
    value_name : str
        Nom de la colonne métier.

    Retour
    ------
    pandas.DataFrame
        Dimension avec clé substitut et valeur métier.
    """

    # Nous extrayons les valeurs non nulles, uniques et triées.
    unique_values = sorted(v for v in values.dropna().unique())

    # Nous créons le DataFrame métier.
    dim = pd.DataFrame({value_name: unique_values})

    # Nous créons une clé substitut commençant à 1.
    dim[key_name] = range(1, len(dim) + 1)

    # Nous retournons la clé puis la valeur.
    return dim[[key_name, value_name]]

## Étape 3 - Construire les dimensions

In [5]:
# Nous construisons la dimension date.
dim_date = build_dim_date(tr_notice_kpi_base["publication_date"])

# Nous construisons la dimension acheteur.
dim_buyer = build_simple_dimension(tr_notice_kpi_base["buyer_name"], "buyer_key", "buyer_name")

# Nous construisons la dimension type de marché.
dim_market_type = build_simple_dimension(tr_notice_kpi_base["market_type_code"], "market_type_key", "market_type_code")

# Nous construisons la dimension CPV.
dim_cpv = build_simple_dimension(tr_notice_cpv["cpv_code"], "cpv_key", "cpv_code")

# Nous affichons les dimensions.
display(dim_date)
display(dim_buyer)
display(dim_market_type)
display(dim_cpv)

,date_key,full_date,year,month,day,month_label
0,20150303,2015-03-03,2015,3,3,2015-03
1,20150304,2015-03-04,2015,3,4,2015-03
2,20150305,2015-03-05,2015,3,5,2015-03
3,20150306,2015-03-06,2015,3,6,2015-03
4,20150307,2015-03-07,2015,3,7,2015-03
5,20150308,2015-03-08,2015,3,8,2015-03


,buyer_key,buyer_name
0,1,Communauté urbaine de Bordeaux
1,2,Conseil départemental de Loire-Atlantique
2,3,Cté Cnes Moyenne Vilaine et Semnon
3,4,Métropole de Lyon
4,5,Ville de Nantes
5,6,Ville de Rennes


,market_type_key,market_type_code
0,1,FOURNITURES
1,2,SERVICES
2,3,TRAVAUX


,cpv_key,cpv_code
0,1,30200000
1,2,34992200
2,3,45111291
3,4,45233140
4,5,45453100
5,6,48000000
6,7,50700000
7,8,70122100
8,9,71000000
9,10,79111000


## Étape 4 - Construire `fact_notice_publication`

### Grain

**1 ligne = 1 avis retenu pour l'analyse**

### Mesures conservées

- `notice_count`
- `response_delay_days`

### Flags conservés

- `has_deadline_flag`

### Pourquoi ce choix

C'est petit, mais suffisant pour les KPI.  
On évite d'encombrer la fact avec des éléments inutiles.

In [6]:
# Nous copions la base transformée pour construire la fact.
fact_notice = tr_notice_kpi_base.copy()

# Nous ajoutons une version timestamp de la date pour la jointure avec la dimension date.
fact_notice["publication_date_ts"] = pd.to_datetime(fact_notice["publication_date"])

# Nous joignons la clé de la dimension date.
fact_notice = fact_notice.merge(
    dim_date[["date_key", "full_date"]],
    left_on="publication_date_ts",
    right_on="full_date",
    how="left",
)

# Nous joignons la clé de la dimension acheteur.
fact_notice = fact_notice.merge(dim_buyer, on="buyer_name", how="left")

# Nous joignons la clé de la dimension type de marché.
fact_notice = fact_notice.merge(dim_market_type, on="market_type_code", how="left")

# Nous créons une clé technique de fact.
fact_notice["fact_notice_key"] = range(1, len(fact_notice) + 1)

# Nous sélectionnons seulement les colonnes finales de la fact.
fact_notice_publication = fact_notice[
    [
        "fact_notice_key",
        "idweb",
        "date_key",
        "buyer_key",
        "market_type_key",
        "notice_count",
        "response_delay_days",
        "has_deadline_flag",
        "batch_id",
        "url_avis",
        "objet_court",
    ]
].rename(columns={"idweb": "notice_business_id"})

# Nous affichons la fact finale.
fact_notice_publication

,fact_notice_key,notice_business_id,date_key,buyer_key,market_type_key,notice_count,response_delay_days,has_deadline_flag,batch_id,url_avis,objet_court
0,1,15-31501,20150303,3,3,1,48,True,B20150308_01_initial,https://www.boamp.fr/pages/avis/?q=idweb:15-31501,concession d'aménagement pour la réalisation d...
1,2,15-40000,20150304,6,3,1,14,True,B20150308_01_initial,https://www.boamp.fr/pages/avis/?q=idweb:15-40000,maintenance de bâtiments scolaires et travaux ...
2,3,15-40001,20150305,5,1,1,20,True,B20150308_01_initial,https://www.boamp.fr/pages/avis/?q=idweb:15-40001,fourniture de matériel informatique pour les s...
3,4,15-40002,20150306,2,2,1,14,True,B20150308_01_initial,https://www.boamp.fr/pages/avis/?q=idweb:15-40002,prestations de nettoyage des locaux administra...
4,5,15-40003,20150307,4,3,1,23,True,B20150309_01_incremental,https://www.boamp.fr/pages/avis/?q=idweb:15-40003,travaux d'aménagement de voirie et signalisation
5,6,15-40004,20150308,1,2,1,14,True,B20150309_01_incremental,https://www.boamp.fr/pages/avis/?q=idweb:15-40004,services d'assistance juridique en urbanisme


## Étape 5 - Construire `fact_notice_cpv`

### Pourquoi une deuxième fact

Un avis peut avoir plusieurs CPV.

Si on mettait les CPV directement dans `fact_notice_publication` :

- on dupliquerait les lignes,
- on casserait le grain,
- on fausserait les comptages.

On crée donc une petite fact de liaison analytique :

- une ligne par couple `notice + cpv`

In [7]:
# Nous joignons la clé CPV.
fact_notice_cpv = tr_notice_cpv.merge(dim_cpv, on="cpv_code", how="left")

# Nous joignons la clé de fact notice à partir de l'identifiant métier.
fact_notice_cpv = fact_notice_cpv.merge(
    fact_notice_publication[["fact_notice_key", "notice_business_id", "date_key"]],
    left_on="idweb",
    right_on="notice_business_id",
    how="left",
)

# Nous sélectionnons les colonnes finales.
fact_notice_cpv = fact_notice_cpv[
    [
        "fact_notice_key",
        "notice_business_id",
        "date_key",
        "cpv_key",
        "is_primary_flag",
    ]
].drop_duplicates().reset_index(drop=True)

# Nous affichons la fact CPV.
fact_notice_cpv

,fact_notice_key,notice_business_id,date_key,cpv_key,is_primary_flag
0,1,15-31501,20150303,3,True
1,1,15-31501,20150303,9,False
2,1,15-31501,20150303,8,False
3,2,15-40000,20150304,5,True
4,2,15-40000,20150304,7,False
5,3,15-40001,20150305,1,True
6,3,15-40001,20150305,6,False
7,4,15-40002,20150306,11,True
8,5,15-40003,20150307,4,True
9,5,15-40003,20150307,2,False


## Étape 6 - Sauvegarder le DWH

In [8]:
# Nous définissons le dossier warehouse.
warehouse_dir = output_dir / "warehouse"

# Nous créons le dossier si besoin.
warehouse_dir.mkdir(parents=True, exist_ok=True)

# Nous écrivons toutes les dimensions.
dim_date.to_csv(warehouse_dir / "dim_date.csv", index=False)
dim_buyer.to_csv(warehouse_dir / "dim_buyer.csv", index=False)
dim_market_type.to_csv(warehouse_dir / "dim_market_type.csv", index=False)
dim_cpv.to_csv(warehouse_dir / "dim_cpv.csv", index=False)

# Nous écrivons toutes les facts.
fact_notice_publication.to_csv(warehouse_dir / "fact_notice_publication.csv", index=False)
fact_notice_cpv.to_csv(warehouse_dir / "fact_notice_cpv.csv", index=False)

# Nous affichons la liste finale.
sorted(p.name for p in warehouse_dir.glob("*.csv"))

['dim_buyer.csv',
 'dim_cpv.csv',
 'dim_date.csv',
 'dim_market_type.csv',
 'fact_notice_cpv.csv',
 'fact_notice_publication.csv']

## Exemple de sauvegarde dans une base de données PostgreSQL

In [10]:
!pip install sqlalchemy

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 2.6 MB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 2.9 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 3.1 MB/s  0:00:00



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Bilan 

### Ce qu'on a volontairement **retiré**

- pas de dimension département,
- pas de dimension procédure,
- pas de dimension nature d'avis,
- pas de dimension lot,
- pas de grosses colonnes textuelles dans la fact,
- pas de modèle "tout BOAMP".

### Pourquoi 

> **réduire le modèle à ce qui sert vraiment les KPI**

